In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 17.0 MB/s eta 0:00:00


In [ ]:
import SimpleITK as sitk

# Read the NIfTI file (.nii or .nii.gz)
image = sitk.ReadImage("/content/drive/MyDrive/drr_generation_files/ct_01.nii.gz")

# Write as MetaImage (.mhd + .raw)
sitk.WriteImage(image, "output.mhd")


In [ ]:
import nibabel as nib
from nibabel.orientations import axcodes2ornt, ornt_transform, io_orientation, apply_orientation

# ---- User inputs ----
input_nii = "/content/drive/MyDrive/drr_generation_files/ct_01.nii.gz"
output_nii = "output_RAI.nii.gz"
target_orientation = "RAI"

# ---- Load image ----
img = nib.load(input_nii)
data = img.get_fdata()
affine = img.affine

print("affine ",affine)

# ---- Get current orientation ----
current_ornt = io_orientation(affine)
print("current_ornt",current_ornt)
print("Current orientation:", nib.orientations.ornt2axcodes(current_ornt))

# ---- Compute target orientation ----
target_ornt = axcodes2ornt(tuple(target_orientation))
print("Target orientation ",target_ornt)
# ---- Compute transform ----
transform = ornt_transform(current_ornt, target_ornt)
print("Transform ",transform)
# ---- Apply transform ----
reoriented_data = apply_orientation(data, transform)
new_affine = nib.orientations.inv_ornt_aff(transform, data.shape)
print("new_affine",new_affine)
new_img = nib.Nifti1Image(reoriented_data, affine @ new_affine)

# ---- Save reoriented image ----
nib.save(new_img, output_nii)
print(f"✅ Saved reoriented image to {output_nii} (orientation: {target_orientation})")


affine  [[-0.703125  0.        0.       -0.      ]
 [ 0.       -0.703125  0.       -0.      ]
 [ 0.        0.        1.25      0.      ]
 [ 0.        0.        0.        1.      ]]
current_ornt [[ 0. -1.]
 [ 1. -1.]
 [ 2.  1.]]
Current orientation: ('L', 'P', 'S')
Target orientation  [[ 0.  1.]
 [ 1.  1.]
 [ 2. -1.]]
Transform  [[ 0. -1.]
 [ 1. -1.]
 [ 2. -1.]]
new_affine [[ -1.   0.   0. 511.]
 [  0.  -1.   0. 511.]
 [  0.   0.  -1. 231.]
 [  0.   0.   0.   1.]]
✅ Saved reoriented image to output_RAI.nii.gz (orientation: RAI)


In [ ]:
import nibabel as nib
from nibabel.orientations import axcodes2ornt, ornt_transform, io_orientation, apply_orientation

# ---- User inputs ----
input_nii = "/content/drive/MyDrive/drr_generation_files/ct_01.nii.gz"

# ---- Load image ----
img = nib.load(input_nii)
data = img.get_fdata()
affine = img.affine

print("affine ",affine)

# ---- Get current orientation ----
current_ornt = io_orientation(affine)
print("current_ornt",current_ornt)
print("Current orientation:", nib.orientations.ornt2axcodes(current_ornt))


affine  [[-0.703125  0.        0.       -0.      ]
 [ 0.       -0.703125  0.       -0.      ]
 [ 0.        0.        1.25      0.      ]
 [ 0.        0.        0.        1.      ]]
current_ornt [[ 0. -1.]
 [ 1. -1.]
 [ 2.  1.]]
Current orientation: ('L', 'P', 'S')


In [ ]:
import SimpleITK as sitk
import nibabel as nib
import numpy as np
from nibabel.orientations import axcodes2ornt, io_orientation, ornt_transform, apply_orientation, inv_ornt_aff

def mhd_to_nifti(input_mhd, output_nii):
    """
    Convert MetaImage (.mhd/.raw) to NIfTI (.nii.gz), reorienting from LPS → RAS.
    Works safely with ITK and neuroimaging coordinate differences.
    """
    print(f"🔹 Reading MHD file: {input_mhd}")
    # --- Read MHD with SimpleITK ---
    itk_img = sitk.ReadImage(input_mhd)
    data = sitk.GetArrayFromImage(itk_img)      # numpy array (z, y, x)
    spacing = itk_img.GetSpacing()
    origin = itk_img.GetOrigin()
    direction = np.array(itk_img.GetDirection()).reshape(3, 3)

    print("📏 Spacing:", spacing)
    print("📍 Origin:", origin)
    print("🧭 Direction (ITK, LPS-based):\n", direction)

    # --- Build affine for NIfTI (RAS world) ---
    # ITK uses LPS → flip X and Y to get RAS
    lps_to_ras = np.diag([-1, -1, 1])
    ras_direction = lps_to_ras @ direction @ lps_to_ras
    ras_origin = lps_to_ras @ np.array(origin)
    affine = np.eye(4)
    affine[:3, :3] = ras_direction * np.array(spacing)
    affine[:3, 3] = ras_origin

    # --- Reorder data axes (SimpleITK gives z,y,x) ---
    data = np.transpose(data, (2, 1, 0))

    # --- Create and save NIfTI ---
    nii_img = nib.Nifti1Image(data, affine)
    nib.save(nii_img, output_nii)

    print(f"✅ Saved NIfTI (RAS-oriented): {output_nii}")
    print("Orientation (NIfTI world):", nib.aff2axcodes(nii_img.affine))

    return output_nii


In [ ]:
mhd_to_nifti(
    input_mhd="/content/drive/MyDrive/drr_generation_files/sub-verse601_img.mhd",
    output_nii="ct_01_converted.nii.gz"
)


🔹 Reading MHD file: /content/drive/MyDrive/drr_generation_files/sub-verse601_img.mhd
📏 Spacing: (0.902344, 0.902344, 0.9)
📍 Origin: (-206.052, 333.898, -672.59)
🧭 Direction (ITK, LPS-based):
 [[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]
✅ Saved NIfTI (RAS-oriented): ct_01_converted.nii.gz
Orientation (NIfTI world): ('R', 'P', 'S')


'ct_01_converted.nii.gz'

In [ ]:
import nibabel as nib
import SimpleITK as sitk
import numpy as np
from nibabel.orientations import io_orientation, axcodes2ornt, ornt_transform, apply_orientation, inv_ornt_aff

def nifti_to_mhd(input_nii, output_mhd):
    """
    Convert NIfTI (.nii/.nii.gz) → MetaImage (.mhd/.raw),
    safely reorienting from RAS (NIfTI world) → LPS (ITK world).
    """

    print(f"🔹 Reading NIfTI file: {input_nii}")
    nii = nib.load(input_nii)
    data = nii.get_fdata()
    affine = nii.affine

    # --- Detect current orientation (e.g., ('R', 'A', 'S')) ---
    current_ornt = io_orientation(affine)
    current_axcodes = nib.orientations.ornt2axcodes(current_ornt)
    print("🧭 Current orientation:", current_axcodes)

    # --- Define target orientation (ITK uses LPS) ---
    target_axcodes = ('L', 'P', 'S')
    target_ornt = axcodes2ornt(target_axcodes)
    transform = ornt_transform(current_ornt, target_ornt)

    # --- Apply orientation transform ---
    reoriented_data = apply_orientation(data, transform)
    new_affine = affine @ inv_ornt_aff(transform, data.shape)

    # --- Create temporary NIfTI aligned to LPS ---
    temp_nii = "temp_LPS_oriented.nii.gz"
    nib.save(nib.Nifti1Image(reoriented_data, new_affine), temp_nii)
    print(f"✅ Reoriented to LPS (ITK standard): {temp_nii}")

    # --- Read reoriented NIfTI with SimpleITK and write as MHD ---
    sitk_img = sitk.ReadImage(temp_nii)
    sitk.WriteImage(sitk_img, output_mhd)
    print(f"💾 Saved MetaImage file: {output_mhd}")

    # --- Verify output ---
    itk_img = sitk.ReadImage(output_mhd)
    direction = np.array(itk_img.GetDirection()).reshape(3, 3)
    print("📘 MHD direction matrix:\n", direction)

    return output_mhd


In [ ]:
nifti_to_mhd(
    input_nii="/content/drive/MyDrive/drr_generation_files/ct_01.nii.gz",
    output_mhd="ct_01_converted.mhd"
)


🔹 Reading NIfTI file: /content/drive/MyDrive/drr_generation_files/ct_01.nii.gz
🧭 Current orientation: ('L', 'P', 'S')
✅ Reoriented to LPS (ITK standard): temp_LPS_oriented.nii.gz
💾 Saved MetaImage file: ct_01_converted.mhd
📘 MHD direction matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


'ct_01_converted.mhd'